## Signal Transformation 

Information about the different signal transformation techniques goes here: 
resource: https://github.com/birth-outcomes/ctg_exploratory

Applying daydulo et. al signal transformation technique: 
resource: https://link.springer.com/article/10.1186/s12911-022-02068-1



import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ssqueezepy import Wavelet, cwt
import os

In [2]:
# necessary imports
%pip install ssqueezepy
from dataclasses import dataclass
from IPython.display import Image
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from scipy import signal
from sklearn.model_selection import train_test_split
from ssqueezepy import Wavelet, cwt
from ssqueezepy.utils import make_scales, cwt_scalebounds
from ssqueezepy.visuals import imshow

Note: you may need to restart the kernel to use updated packages.


In [3]:
''' 
Step 1: Define input folder path
'''
input_folder = '../cleaned_data/clean_csv'
output_folder = './outputs/spectograms'
labels_path = "../ExpertAnnotations/CTG_Majority_Vote_Labels_FINAL.csv"

# Create output folder if it doesn't exist
os.makedirs(output_folder, exist_ok=True)

### Seperate by Class Using Majority Voting Outcomes:
Normal Class and abnormal class 

Based on 9 expert clinicians opinions, the most agreed outcomes are presented

Normal class: 426

abnormal: 126



In [11]:
labels_df = pd.read_csv("../ExpertAnnotations/CTG_Majority_Vote_Labels_FINAL.csv")
labels_df["binary_label"] = labels_df["Majority_Vote_Label"].map({1: 0, 2: 1})
labels_df = labels_df.dropna(subset=["binary_label"])

In [12]:
labels_df["binary_label"] = labels_df["binary_label"].astype(int)
counts = labels_df["binary_label"].map({0: "normal", 1: "abnormal"}).value_counts()

In [13]:
print("Counts:\n", counts)
print("\nPercentages:\n", (counts / counts.sum() * 100).round(2))

Counts:
 normal      426
abnormal    126
Name: binary_label, dtype: int64

Percentages:
 normal      77.17
abnormal    22.83
Name: binary_label, dtype: float64


### ****Signal pre-processing

Method from Daydulo et al. 2022

Signal pre-processing:

Long gaps (more than 15s) removed from the signal

Missing values at beginning and end of recording excluded to start from the stable point

Outside 50bpm or 200bpm are outliers

Outliers and small gaps were find and linearly interpolated using Matlab

Spikes are when beat is more than 25 from previous adjacent beat (not physiologic, unreliable) so removed using cubic spline interpolation.

Selected segment of the first 20 minutes (4800) and last 15 minutes (3600) to use and represent first and second stage of labour.

**CODE FROM: https://github.com/birth-outcomes/ctg_exploratory/blob/main/08_apply_daydulo_cwt.ipynb**

In [ ]:
def daydulo_clean(fhr):
    '''
    Cleans fetal heart rate (FHR) signal according to Daydulo et al. 2022
    Inputs:
    fhr - series, the "FHR" column from one of the csv files
    '''

    # Replace 0 with NaN
    fhr.replace(0, np.nan, inplace=True)

    # Remove NaN if they occured for more than 15 seconds consecutively
    na = fhr.isnull()
    fhr = fhr[~(fhr.groupby(na.ne(na.shift()).cumsum().values).transform('size').ge(61) & na)].reset_index(drop=True)

    # Set outliers to NaN
    fhr[fhr < 50] = np.nan
    fhr[fhr > 200] = np.nan

    # Replace missing values using linear interpolation
    fhr = fhr.interpolate(method='linear')

    # Find how each value has changed from the prior value
    diff = fhr - fhr.shift()

    # Where difference is more than +- 25, set as NaN
    fhr[(diff > 25) | (diff < -25)] = np.nan

    # Replace missing values using cubic interpolation
    fhr = fhr.interpolate(method='cubic')

    return(fhr)

In [ ]:
def clean_signals(sig_dict):
    '''
    Cleans signals in provided dictionary, and returns two (one representing
    first stage and one representing second stage)
    Inputs:
    - sig_dict - dictionary of dataframes which have FHR column for cleaning
    Outputs:
    - fhr_first - dictionary of clean FHR signals for first stage of labour
    - fhr_second - dictionary of clean FHR signals for second stage of labour
    '''
    fhr_first = dict()
    fhr_second = dict()
    for key, value in sig_dict.items():
        df = daydulo_clean(value.FHR)
        fhr_first[key] = df.head(4800)
        fhr_second[key] = df.tail(3600).reset_index(drop=True)

    return(fhr_first, fhr_second)

In [ ]:
# Apply cleaning to the signals
fhr_normal_first, fhr_normal_second = clean_signals(csv_normal)
fhr_distress_first, fhr_distress_second = clean_signals(csv_distress)

In [ ]:
# Augment distressed records by taking overlapping slices and add to dictionaries
# Use raw data and apply the same cleaning function before slicing
if 'fhr_distress_first' not in globals():
    fhr_distress_first = dict()
if 'fhr_distress_second' not in globals():
    fhr_distress_second = dict()
for key, value in csv_distress.items():
    df = daydulo_clean(value.FHR)
    fhr_distress_first[f'{key}_5'] = df[1200:6000].reset_index(drop=True)
    fhr_distress_first[f'{key}_10'] = df[2400:7200].reset_index(drop=True)
    fhr_distress_second[f'{key}_5'] = df[-4800:-1200].reset_index(drop=True)
    fhr_distress_second[f'{key}_10'] = df[-6000:-2400].reset_index(drop=True)

# Print totals after augmentation
print(f"Total normal signals: {len(fhr_normal_first) + len(fhr_normal_second)}")
print(f"Total distressed signals: {len(fhr_distress_first) + len(fhr_distress_second)}")
print(f"Total records: {len(fhr_normal_first) + len(fhr_normal_second) + len(fhr_distress_first) + len(fhr_distress_second)}")